In [8]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

print("TRANSFER LEARNING - IMAGE CLASSIFIER")
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training images: {X_train.shape[0]}")
print(f"Test images: {X_test.shape[0]}")
print(f"Image shape: {X_train.shape[1:]}")
print(f"Classes: {len(class_names)}")
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

y_train = y_train.flatten()
y_test = y_test.flatten()
train_subset = 5000
test_subset = 1000

X_train_sub = X_train[:train_subset]
y_train_sub = y_train[:train_subset]
X_test_sub = X_test[:test_subset]
y_test_sub = y_test[:test_subset]

print(f"\nUsing subset for faster training:")
print(f"Training: {train_subset} images")
print(f"Testing: {test_subset} images")
print("\nClass distribution in training subset:")
for i, name in enumerate(class_names):
    count = np.sum(y_train_sub == i)
    print(f"  {name}: {count}")

TRANSFER LEARNING - IMAGE CLASSIFIER
Training images: 50000
Test images: 10000
Image shape: (32, 32, 3)
Classes: 10

Using subset for faster training:
Training: 5000 images
Testing: 1000 images

Class distribution in training subset:
  airplane: 505
  automobile: 460
  bird: 519
  cat: 486
  deer: 519
  dog: 488
  frog: 519
  horse: 486
  ship: 520
  truck: 498


In [9]:
print("\nBUILDING TRANSFER LEARNING MODEL")

print("""
Transfer Learning Strategy:
1. Load MobileNetV2 (pretrained on ImageNet -
   1.4 million images, 1000 categories)
2. FREEZE its learned weights (dont retrain them)
3. Add NEW layers on top for our 10 categories
4. Only train the NEW layers (fast!)
""")
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)
base_model.trainable = False

print(f"Base model layers: {len(base_model.layers)}")
print(f"Base model trainable: {base_model.trainable}")
model = tf.keras.Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel Summary:")
model.summary()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = sum([tf.size(w).numpy() for w in model.non_trainable_weights])

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Non-trainable (frozen) parameters: {non_trainable_params:,}")
print(f"We only train {trainable_params/(trainable_params+non_trainable_params)*100:.1f}% of total parameters!")


BUILDING TRANSFER LEARNING MODEL

Transfer Learning Strategy:
1. Load MobileNetV2 (pretrained on ImageNet -
   1.4 million images, 1000 categories)
2. FREEZE its learned weights (dont retrain them)
3. Add NEW layers on top for our 10 categories
4. Only train the NEW layers (fast!)



/tmp/ipykernel_500/4128433954.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Base model layers: 154
Base model trainable: False

Model Summary:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


Trainable parameters: 165,258
Non-trainable (frozen) parameters: 2,257,984
We only train 6.8% of total parameters!


In [11]:
print("\nTRAINING MODEL")

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("Training started (this may take 3-5 minutes)...")

history = model.fit(
    X_train_sub, y_train_sub,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

epochs_trained = len(history.history['loss'])
print(f"Training completed in {epochs_trained} epochs")
test_loss, test_acc = model.evaluate(X_test_sub, y_test_sub, verbose=0)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

train_final_loss = history.history['loss'][-1]
val_final_loss = history.history['val_loss'][-1]
print(f"Final Training Loss: {train_final_loss:.4f}")
print(f"Final Validation Loss: {val_final_loss:.4f}")

diff = abs(val_final_loss - train_final_loss)
if diff > 0.3:
    print("Status: Some overfitting detected")
else:
    print("Status: Good generalization")
print(f"\nComparison:")
print(f"Transfer Learning: {test_acc*100:.2f}% accuracy in {epochs_trained} epochs")
print(f"From-scratch CNN would typically need 50-100+ epochs")
print(f"and much more data to reach similar accuracy")



TRAINING MODEL
Training started (this may take 3-5 minutes)...
Training completed in 9 epochs

Test Accuracy: 31.10%
Final Training Loss: 1.6204
Final Validation Loss: 1.9722
Status: Some overfitting detected

Comparison:
Transfer Learning: 31.10% accuracy in 9 epochs
From-scratch CNN would typically need 50-100+ epochs
and much more data to reach similar accuracy


In [14]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("FIXING THE RESOLUTION MISMATCH")
X_train_resized = tf.image.resize(X_train_sub, (96, 96)).numpy()
X_test_resized = tf.image.resize(X_test_sub, (96, 96)).numpy()

print(f"Original shape: {X_train_sub.shape}")
print(f"Resized shape: {X_train_resized.shape}")
base_model_fixed = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)
base_model_fixed.trainable = False

model_fixed = tf.keras.Sequential([
    base_model_fixed,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model_fixed.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nRetraining with correct image size...")
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_fixed = model_fixed.fit(
    X_train_resized, y_train_sub,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

test_loss_fixed, test_acc_fixed = model_fixed.evaluate(X_test_resized, y_test_sub, verbose=0)
print(f"\nFixed Model Test Accuracy: {test_acc_fixed*100:.2f}%")
print(f"Previous (broken 32x32) accuracy: 29.50%")
print(f"Improvement: +{(test_acc_fixed-0.295)*100:.2f}%")

FIXING THE RESOLUTION MISMATCH
Original shape: (5000, 32, 32, 3)
Resized shape: (5000, 96, 96, 3)

Retraining with correct image size...

Fixed Model Test Accuracy: 77.40%
Previous (broken 32x32) accuracy: 29.50%
Improvement: +47.90%


In [15]:
readme_content = """# Image Classifier - Transfer Learning

## Project Overview
Multi-class image classifier using Transfer Learning with
MobileNetV2, classifying images into 10 CIFAR-10 categories.

## Approach
- Used pretrained MobileNetV2 (trained on ImageNet, 1.4M images)
- Froze base model weights, added custom classification head
- Only trained ~165K parameters (6.8% of total) instead of
  training from scratch

## Debugging Journey (Key Learning)
Initial attempt used MobileNetV2 with CIFAR-10's native 32x32
images, resulting in only 29.5% accuracy. Investigation revealed
MobileNetV2 requires minimum 96x96 input size for its pretrained
weights to transfer meaningfully. After resizing images to 96x96
and rebuilding the model with correct input_shape, accuracy
improved to 77.40% — a 47.9 percentage point gain.

## Results
- Final Test Accuracy: 77.40%
- Improvement from initial (broken) version: +47.90%
- Trainable params: 165,258 (6.8% of total 2.4M)

## Key Techniques
1. Transfer Learning (MobileNetV2 base)
2. Proper input size matching (critical lesson!)
3. Global Average Pooling
4. Dropout regularization
5. Early Stopping

## Tools Used
- TensorFlow/Keras
- MobileNetV2 (Transfer Learning)
- CIFAR-10 dataset

## Author
Kaviya V | github.com/kaviyavijayan11
"""

with open("README.md", "w") as f:
    f.write(readme_content)

print("README.md created!")

README.md created!
